# Proyecto - Implementación Fase 2
## Modelos DL
- Fabiola Contreras -22787
- María Villafuerte -22129

## Preparar el ambiente

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from torchvision import transforms

# Importa desde tu archivo python (donde dejaste modelo y funciones)
from cnn_model import (
    QRDataset, QRCNN, train_one_epoch, evaluate, SEED
)

In [2]:
#  Dispositivo 
device = (
    torch.device("cuda")  if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Dispositivo: {device}")

Dispositivo: cpu


In [3]:
# Datos
META_CSV = "data_procesada/qr_full_dataset.csv"

# Entrenamiento
EPOCHS   = 30
BATCH    = 64
LR       = 1e-3
DROPOUT  = 0.4
PATIENCE = 7

# Salida
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)


## Cargar datos

In [4]:
train_df = pd.read_csv("data_procesada/train.csv")
val_df   = pd.read_csv("data_procesada/val.csv")
test_df  = pd.read_csv("data_procesada/test.csv")

train_paths = train_df["filepath"].tolist()
train_labels = train_df["label"].tolist()

val_paths = val_df["filepath"].tolist()
val_labels = val_df["label"].tolist()

test_paths = test_df["filepath"].tolist()
test_labels = test_df["label"].tolist()

print(len(train_paths), len(val_paths), len(test_paths))

140000 30000 30000


In [5]:
aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
])

ds_train = QRDataset(train_paths, train_labels, transform=aug)
ds_val   = QRDataset(val_paths, val_labels)
ds_test  = QRDataset(test_paths, test_labels)

print(f"Train: {len(ds_train)} | Val: {len(ds_val)} | Test: {len(ds_test)}")

Train: 140000 | Val: 30000 | Test: 30000


In [8]:
kw = dict(num_workers=4, pin_memory=(device.type == "cuda"))

train_loader = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  **kw)
val_loader   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, **kw)
test_loader  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False, **kw)

## Modelo CNN

In [9]:
model = QRCNN(dropout=DROPOUT).to(device)

print(f"Parámetros: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Parámetros: 3142242


### Entrenamiento

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss = float("inf")
patience_cnt  = 0
history       = []

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    vl_loss, vl_acc, vl_auc, _, _ = evaluate(model, val_loader, criterion, device)

    scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": tr_loss,
        "train_acc": tr_acc,
        "val_loss": vl_loss,
        "val_acc": vl_acc,
        "val_auc": vl_auc
    })

    print(epoch, tr_loss, vl_loss)

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_cnt = 0
        torch.save(model.state_dict(), OUT_DIR / "best_model.pt")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"Early stopping en época {epoch}")
            break

### Evaluación

In [ ]:
model.load_state_dict(torch.load(OUT_DIR / "best_model.pt", map_location=device))

te_loss, te_acc, te_auc, preds, true = evaluate(model, test_loader, criterion, device)

print(f"Test Loss: {te_loss}")
print(f"Test Acc: {te_acc}")
print(f"Test AUC: {te_auc}")

print(classification_report(true, preds, target_names=["normal", "malware"]))
print(confusion_matrix(true, preds))


### Guardar resultados

In [ ]:
pd.DataFrame(history).to_csv(OUT_DIR / "training_cnn_history.csv", index=False)

with open(OUT_DIR / "test_metrics.json", "w") as f:
    json.dump({
        "test_loss": te_loss,
        "test_acc": te_acc,
        "test_auc": te_auc
    }, f, indent=2)

print(f"Artefactos guardados en {OUT_DIR}")